# 04 Berika med länkade data

Webperfs data om kommuner och regioner innehåller betyg men inte mycket mer. För att kunna jämföra behöver vi veta mer om varje kommun, till exempel vilken kommunkod den har, vilket län den ligger i och hur många som bor där.

Det hämtar vi från [Wikidata](https://www.wikidata.org), en öppen kunskapsbas där varje sak har ett eget id-nummer som börjar med Q. Webperf länkar till det id-numret i fältet `same_as`. Det kallas länkade data. Två källor som pekar på samma id kan kopplas ihop utan att vi behöver jämföra namn.

Wikidata är licensierad under CC0, alltså fri att använda utan villkor.

In [ ]:
import json
from pathlib import Path

import httpx
import pandas as pd

## Läs in Webperfs data

I notebook 03 sparade vi Webperfs kommuner och regioner i mappen `data/radata`. Vi läser in filerna igen och bygger samma tabell som där. Saknas filerna behöver vi först köra notebook 03.

`json.loads` gör om JSON-text till Pythons ordböcker, alltså motsatsen till `json.dumps` som vi använde när vi sparade.

In [ ]:
RADATA = Path("..") / "data" / "radata"

kommuner_fil = json.loads((RADATA / "webperf-kommuner.json").read_text(encoding="utf-8"))
regioner_fil = json.loads((RADATA / "webperf-regioner.json").read_text(encoding="utf-8"))

kommuner = pd.json_normalize(kommuner_fil["svar"]["municipalities"])
regioner = pd.json_normalize(regioner_fil["svar"]["regions"])
kommuner["typ"] = "kommun"
regioner["typ"] = "region"

webperf = pd.concat([kommuner, regioner], ignore_index=True)
webperf = webperf[["id", "title", "typ", "same_as"]]

webperf.head()

Med dubbla hakparenteser väljer vi ut de kolumner vi behöver här. Betygen tar vi in igen när vi slår ihop källorna i en senare notebook.

## Id-numret i Wikidata

Fältet `same_as` är en hel adress. Vi behöver bara id-numret i slutet, till exempel `Q498470`. Med `str.rsplit("/", n=1)` delar vi texten vid det sista snedstrecket och med `str[-1]` tar vi den sista delen.

In [ ]:
webperf["qid"] = webperf["same_as"].str.rsplit("/", n=1).str[-1]

webperf[["title", "same_as", "qid"]].head()

## Att fråga Wikidata

Wikidata besvarar frågor skrivna i SPARQL, ett frågespråk för länkade data. En fråga beskriver ett mönster och Wikidata letar upp allt som passar.

Allt i Wikidata är uppbyggt av påståenden med tre delar, en sak, en egenskap och ett värde. Påståendet att Ale kommun har kommunkoden 1440 skrivs `wd:Q498470 wdt:P525 "1440"`. Här är `Q498470` Ale kommun och `P525` egenskapen kommunkod i Sverige. Vill vi veta värdet ersätter vi det med en variabel som börjar med frågetecken, till exempel `?kommunkod`.

Vi skriver en funktion som skickar en fråga och gör om svaret till en tabell. Wikidata kräver att vi talar om vem vi är i headern `User-Agent`. Utan den svarar Wikidata med statuskod 403. Frågan skickas med POST, eftersom den blir lång när vi frågar om alla kommuner på en gång.

Svaret har en rad per träff, där varje variabel har ett värde. Vi plockar ut värdena och lägger dem i en tabell.

In [ ]:
WIKIDATA = "https://query.wikidata.org/sparql"
HEADERS = {"User-Agent": "webbanalys.se, kodexempel i serien Python for webbanalys (https://webbanalys.se)"}


def fraga_wikidata(sparql):
    svar = httpx.post(WIKIDATA, data={"query": sparql, "format": "json"}, headers=HEADERS, timeout=120)
    svar.raise_for_status()
    innehall = svar.json()
    rader = []
    for traff in innehall["results"]["bindings"]:
        rader.append({namn: varde["value"] for namn, varde in traff.items()})
    return pd.DataFrame(rader), innehall

Funktionen lämnar tillbaka två saker, tabellen och hela svaret. Hela svaret sparar vi som rådata längre ned.

Vi provar med en enda kommun. Raden med `SERVICE wikibase:label` ger oss namnet på länet på svenska. Variabeln `?lan` blir då id-numret för länet och `?lanLabel` dess namn.

In [ ]:
tabell, _ = fraga_wikidata('''
SELECT ?kommunkod ?lan ?lanLabel WHERE {
  wd:Q498470 wdt:P525 ?kommunkod ;
             wdt:P131 ?lan .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "sv". }
}
''')

tabell

Understrecket `_` är ett vanligt namn för ett värde vi inte tänker använda, här hela svaret.

Semikolonet i frågan betyder att nästa rad gäller samma sak, alltså Ale kommun. Egenskapen `P131` betyder ligger i administrativt område.

## När en kommun har flera kommunkoder

Samma fråga för Trollhättans kommun ger två rader.

In [ ]:
tabell, _ = fraga_wikidata('''
SELECT ?kommunkod WHERE {
  wd:Q28532 wdt:P525 ?kommunkod .
}
''')

tabell

Trollhättan bytte kommunkod 1998 när kommunen flyttades till Västra Götalands län. Wikidata sparar båda koderna. Den gamla har ett slutdatum, en så kallad kvalificerare.

För att se kvalificerare skriver vi frågan på ett längre sätt. `p:P525` ger själva påståendet, `ps:P525` dess värde och `pq:P582` kvalificeraren slutdatum. `OPTIONAL` betyder att raden kommer med även om det inte finns något slutdatum.

In [ ]:
tabell, _ = fraga_wikidata('''
SELECT ?kommunkod ?slutdatum WHERE {
  wd:Q28532 p:P525 ?pastaende .
  ?pastaende ps:P525 ?kommunkod .
  OPTIONAL { ?pastaende pq:P582 ?slutdatum . }
}
''')

tabell

Den aktuella koden är den som saknar slutdatum. Det kan vi uttrycka med `FILTER NOT EXISTS`, som tar bort påståenden där mönstret inom klamrarna finns.

## Kommunkod för alla på en gång

Nu ställer vi frågan för alla webbplatser samtidigt. Med `VALUES` räknar vi upp de id-nummer vi vill fråga om. Vi bygger listan av kolumnen `qid` med `" ".join(...)`, som sätter ihop texterna med mellanslag emellan.

In [ ]:
alla_qid = " ".join("wd:" + qid for qid in webperf["qid"])

alla_qid[:80]

I Python sätter vi in listan i frågan med en f-sträng, en text som börjar med `f`. Där ersätts `{alla_qid}` med innehållet i variabeln. Klamrar som ska finnas kvar i själva frågan skrivs dubbla, `{{` och `}}`.

In [ ]:
FRAGA_KOMMUNKOD = f'''
SELECT ?item ?kommunkod WHERE {{
  VALUES ?item {{ {alla_qid} }}
  ?item p:P525 ?pastaende .
  ?pastaende ps:P525 ?kommunkod .
  FILTER NOT EXISTS {{ ?pastaende pq:P582 ?slutdatum . }}
}}
'''

kommunkoder, svar_kommunkod = fraga_wikidata(FRAGA_KOMMUNKOD)

kommunkoder.shape

Kolumnen `item` innehåller hela adressen till objektet. Vi gör om den till id-nummer på samma sätt som tidigare, så att vi kan koppla ihop tabellerna.

In [ ]:
kommunkoder["qid"] = kommunkoder["item"].str.rsplit("/", n=1).str[-1]
kommunkoder = kommunkoder[["qid", "kommunkod"]]

kommunkoder.head()

Kommunkoden är text och ska förbli text. Koder som `0180` för Stockholm börjar med en nolla, och den skulle försvinna om vi gjorde om koden till ett tal.

Vi kontrollerar att ingen webbplats fick mer än en kod. `duplicated` markerar rader vars värde redan förekommit och `sum` räknar dem.

In [ ]:
kommunkoder["qid"].duplicated().sum()

## Län

Kommuner pekar på sitt län med `P131`. Det gäller inte alla regioner, så vi letar åt båda hållen. Ett län pekar på kommunerna och regionen med egenskapen `P150`, innehåller administrativ enhet. `UNION` betyder att något av de två mönstren räcker.

Vi kräver också att det vi hittar är ett län, alltså en instans, `P31`, av Sveriges län, `Q200547`. Annars kommer till exempel landskap och sjukvårdsregioner med. Från länet hämtar vi länskoden med `P507`.

För de flesta kommuner fungerar båda vägarna, och då kommer samma län med två gånger. `DISTINCT` efter `SELECT` tar bort rader som är exakt likadana.

In [ ]:
FRAGA_LAN = f'''
SELECT DISTINCT ?item ?lanLabel ?lanskod WHERE {{
  VALUES ?item {{ {alla_qid} }}
  {{ ?item wdt:P131 ?lan . }} UNION {{ ?lan wdt:P150 ?item . }}
  ?lan wdt:P31 wd:Q200547 ;
       wdt:P507 ?lanskod .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "sv". }}
}}
'''

lan, svar_lan = fraga_wikidata(FRAGA_LAN)

lan["qid"] = lan["item"].str.rsplit("/", n=1).str[-1]
lan = lan.rename(columns={"lanLabel": "lan"})
lan = lan[["qid", "lan", "lanskod"]]

lan.shape, lan["qid"].duplicated().sum()

Med `rename` byter vi namn på kolumnen `lanLabel` till `lan`.

## Befolkning

Befolkningen ändras över tid, så Wikidata har många värden för varje kommun, ett för varje mättillfälle. Det senaste är oftast markerat som det föredragna. `wikibase:BestRank` väljer det föredragna värdet, eller alla värden om inget är markerat. Kvalificeraren `P585` anger vid vilken tidpunkt befolkningen räknades.

In [ ]:
FRAGA_BEFOLKNING = f'''
SELECT ?item ?befolkning ?tidpunkt WHERE {{
  VALUES ?item {{ {alla_qid} }}
  ?item p:P1082 ?pastaende .
  ?pastaende a wikibase:BestRank ;
             ps:P1082 ?befolkning .
  OPTIONAL {{ ?pastaende pq:P585 ?tidpunkt . }}
}}
'''

befolkning, svar_befolkning = fraga_wikidata(FRAGA_BEFOLKNING)

befolkning["qid"] = befolkning["item"].str.rsplit("/", n=1).str[-1]
befolkning = befolkning[["qid", "befolkning", "tidpunkt"]]

befolkning.shape, befolkning["qid"].duplicated().sum()

Alla värden från Wikidata kommer som text. Befolkningen gör vi om till heltal och tidpunkten till ett datum. Typen `Int64` med stort I är ett heltal som också kan sakna värde, vilket vanliga heltal i pandas inte kan.

In [ ]:
befolkning["befolkning"] = befolkning["befolkning"].astype("Int64")
befolkning["tidpunkt"] = pd.to_datetime(befolkning["tidpunkt"]).dt.date

befolkning.dtypes

`dt.date` behåller bara datumet och tar bort klockslaget, som alltid är midnatt här.

## Slå ihop

Nu kopplar vi ihop de tre svaren med Webperfs tabell via `qid`. Med `how="left"` behåller vi alla rader i Webperfs tabell, även de som saknar en träff i Wikidata. Där blir värdet tomt.

In [ ]:
berikad = (
    webperf
    .merge(kommunkoder, on="qid", how="left")
    .merge(lan, on="qid", how="left")
    .merge(befolkning, on="qid", how="left")
)

berikad.shape

Antalet rader ska vara detsamma som i Webperfs tabell. Fler rader skulle betyda att någon webbplats fått flera träffar.

In [ ]:
len(berikad) == len(webperf)

In [ ]:
berikad.head()

## Vad saknas?

Vi räknar tomma värden per kolumn och delar upp på kommuner och regioner med `groupby`.

In [ ]:
berikad.drop(columns="typ").isna().groupby(berikad["typ"]).sum()

Regionerna har inte kommunkod, och i Wikidata har de inte heller någon befolkning. En region ansvarar för hela länet, så i analysen kan vi använda länets befolkning för regionerna. Det gör vi när vi harmoniserar i notebook 07.

Undantaget är Gotland, som är både kommun och region. Webperf länkar Region Gotland till kommunen i Wikidata, så där finns både kommunkod och befolkning.

In [ ]:
berikad[berikad["title"].str.contains("Gotland")]

Alla kommuner borde ha både kommunkod och befolkning. Vi listar de kommuner där något saknas.

In [ ]:
saknas = berikad[(berikad["typ"] == "kommun") & berikad[["kommunkod", "befolkning"]].isna().any(axis=1)]

saknas[["title", "same_as", "kommunkod", "befolkning"]]

Är listan tom har alla kommuner fått sina värden. Annars öppnar vi länken i `same_as` och kontrollerar att den pekar på rätt objekt. Ett vanligt fel är att länken går till ett äldre objekt med samma namn, till exempel en kommun som bara fanns under några år.

Tecknet `&` betyder och. `any(axis=1)` är sant om något av värdena på raden saknas.

## Spara rådata

Vi sparar de tre svaren från Wikidata som de kom, tillsammans med frågan som gav dem och när vi hämtade dem. Med frågan sparad kan vi senare se exakt vad vi bad om.

In [ ]:
hamtad = pd.Timestamp.now(tz="Europe/Stockholm").isoformat(timespec="seconds")

for namn, fraga, svar in [
    ("kommunkod", FRAGA_KOMMUNKOD, svar_kommunkod),
    ("lan", FRAGA_LAN, svar_lan),
    ("befolkning", FRAGA_BEFOLKNING, svar_befolkning),
]:
    fil = RADATA / f"wikidata-{namn}.json"
    paket = {"hamtad": hamtad, "kalla": WIKIDATA, "fraga": fraga, "svar": svar}
    fil.write_text(json.dumps(paket, ensure_ascii=False, indent=2), encoding="utf-8")
    print(fil)

## Det här vet vi nu

- Webperfs `same_as` ger oss id-numret i Wikidata, och med det kan vi hämta mer om varje kommun och region.
- En fråga i SPARQL beskriver ett mönster av sak, egenskap och värde.
- Wikidata sparar historik. Äldre kommunkoder har ett slutdatum och befolkningen finns för många tidpunkter, så vi behöver välja rätt värde.
- Kommunkoden är text, eftersom den kan börja med en nolla.
- Efter en sammanslagning kontrollerar vi att antalet rader inte har ändrats.

I nästa notebook hämtar vi fältdata om hur snabba webbplatserna är för riktiga besökare, från Chrome UX Report.